In [2]:
import pandas as pd
import numpy as np

In [3]:
filtered_df = pd.read_csv("../outputs_csv/tracking_filtered_by_play_and_frame.csv")

/var/folders/rj/tzk4w23s0md7pcnpq4w2fy6c0000gn/T/ipykernel_42299/1599047575.py:1: DtypeWarning: Columns (0: foulNames, 1: foulIds) have mixed types. Specify dtype option on import or set low_memory=False.
  filtered_df = pd.read_csv("../outputs_csv/tracking_filtered_by_play_and_frame.csv")


In [4]:
filtered_df = filtered_df.drop(['y_LT','y_RT','three_seconds_after','ttt_le_1_5_and_behind_los'],axis=1)

In [5]:
pass_blockers = filtered_df[~filtered_df['pff_blockType'].isna()]
pass_rushers = filtered_df[filtered_df['pff_role'] == 'Pass Rush']

In [6]:


blockers = pass_blockers[
    ['gameId', 'playId', 'frameID', 'nflId', 'x', 'y', 'o']
].copy()

rushers = pass_rushers[
    ['gameId', 'playId', 'frameID', 'nflId', 'x', 'y']
].copy()

blockers = blockers.rename(columns={
    'nflId': 'blocker_nflId',
    'x': 'x_blocker',
    'y': 'y_blocker',
    'o': 'o_blocker'
})

rushers = rushers.rename(columns={
    'nflId': 'rusher_nflId',
    'x': 'x_rusher',
    'y': 'y_rusher'
})

# Every blocker paired with every rusher in the same frame
blocker_rusher_pairs = blockers.merge(
    rushers,
    on=['gameId', 'playId', 'frameID'],
    how='inner'
)

# Vector from blocker to rusher
blocker_rusher_pairs['dx'] = (
    blocker_rusher_pairs['x_rusher'] - blocker_rusher_pairs['x_blocker']
)
blocker_rusher_pairs['dy'] = (
    blocker_rusher_pairs['y_rusher'] - blocker_rusher_pairs['y_blocker']
)

# Euclidean distance
blocker_rusher_pairs['actual_distance'] = np.sqrt(
    blocker_rusher_pairs['dx']**2 + blocker_rusher_pairs['dy']**2
)

# Blocker facing direction unit vector under NFL angle convention:
# 0° = (0,1), 90° = (1,0)
o_rad = np.radians(blocker_rusher_pairs['o_blocker'])
blocker_rusher_pairs['ux_blocker'] = np.sin(o_rad)
blocker_rusher_pairs['uy_blocker'] = np.cos(o_rad)

# Unit vector from blocker to rusher
nonzero_dist = blocker_rusher_pairs['actual_distance'] > 0

blocker_rusher_pairs['ux_to_rusher'] = np.where(
    nonzero_dist,
    blocker_rusher_pairs['dx'] / blocker_rusher_pairs['actual_distance'],
    np.nan
)
blocker_rusher_pairs['uy_to_rusher'] = np.where(
    nonzero_dist,
    blocker_rusher_pairs['dy'] / blocker_rusher_pairs['actual_distance'],
    np.nan
)

# cos(theta) using dot product
blocker_rusher_pairs['cos_theta'] = (
    blocker_rusher_pairs['ux_blocker'] * blocker_rusher_pairs['ux_to_rusher']
    + blocker_rusher_pairs['uy_blocker'] * blocker_rusher_pairs['uy_to_rusher']
).clip(-1, 1)

# Optional: recover theta in degrees
blocker_rusher_pairs['theta_deg'] = np.degrees(
    np.arccos(blocker_rusher_pairs['cos_theta'])
)

# Weighted distance
blocker_rusher_pairs['weighted_distance'] = np.where(
    blocker_rusher_pairs['cos_theta'] > 0,
    blocker_rusher_pairs['actual_distance'] / blocker_rusher_pairs['cos_theta'],
    np.inf
)

# Optional final column selection
blocker_rusher_pairs = blocker_rusher_pairs[
    [
        'gameId', 'playId', 'frameID',
        'blocker_nflId', 'rusher_nflId',
        'x_blocker', 'y_blocker', 'o_blocker',
        'x_rusher', 'y_rusher',
        'dx', 'dy',
        'actual_distance', 'cos_theta', 'theta_deg', 'weighted_distance'
    ]
].copy()

blocker_rusher_pairs

,gameId,playId,frameID,blocker_nflId,rusher_nflId,x_blocker,y_blocker,o_blocker,x_rusher,y_rusher,dx,dy,actual_distance,cos_theta,theta_deg,weighted_distance
0,2021090900,97,11,40151,41263,41.58,24.31,71.03,42.34,19.20,0.76,-5.11,5.166208,-0.182416,100.510525,inf
1,2021090900,97,11,40151,42403,41.58,24.31,71.03,42.71,32.07,1.13,7.76,7.841843,0.457953,62.744905,17.123686
2,2021090900,97,11,40151,44955,41.58,24.31,71.03,42.55,25.37,0.97,1.06,1.436837,0.878246,28.568545,1.636031
3,2021090900,97,11,40151,53441,41.58,24.31,71.03,42.81,22.24,1.23,-2.07,2.407862,0.203623,78.251096,11.825097
4,2021090900,97,11,40151,53504,41.58,24.31,71.03,43.25,26.78,1.67,2.47,2.981577,0.798984,36.966849,3.731712
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3844771,2021110100,4433,36,52507,52585,22.41,26.22,217.53,21.04,20.71,-1.37,-5.51,5.677764,0.916592,23.567181,6.194429
3844772,2021110100,4433,37,52507,42406,22.14,26.15,207.07,21.81,24.27,-0.33,-1.88,1.908743,0.955720,17.114186,1.997178
3844773,2021110100,4433,37,52507,43326,22.14,26.15,207.07,22.17,17.56,0.03,-8.59,8.590052,0.888856,27.270101,9.664162
3844774,2021110100,4433,37,52507,43338,22.14,26.15,207.07,21.73,25.86,-0.41,-0.29,0.502195,0.885738,27.657579,0.566980


In [7]:
blocker_rusher_pairs = blocker_rusher_pairs[
    (blocker_rusher_pairs['weighted_distance'] >= 0) &
    (blocker_rusher_pairs['weighted_distance'] <= 3.5)
].copy()

closest_rusher_per_blocker = (
    blocker_rusher_pairs
    .sort_values(
        ['gameId', 'playId', 'frameID', 'blocker_nflId', 'weighted_distance']
    )
    .drop_duplicates(
        subset=['gameId', 'playId', 'frameID', 'blocker_nflId'],
        keep='first'
    )
    .copy()
)

In [8]:
closest_rusher_per_blocker

,gameId,playId,frameID,blocker_nflId,rusher_nflId,x_blocker,y_blocker,o_blocker,x_rusher,y_rusher,dx,dy,actual_distance,cos_theta,theta_deg,weighted_distance
2,2021090900,97,11,40151,44955,41.58,24.31,71.03,42.55,25.37,0.97,1.06,1.436837,0.878246,28.568545,1.636031
132,2021090900,97,11,42377,44955,40.46,27.30,145.97,42.55,25.37,2.09,-1.93,2.844820,0.973382,13.249226,2.922613
262,2021090900,97,11,42404,44955,41.59,25.67,98.78,42.55,25.37,0.96,-0.30,1.005783,0.988824,8.574025,1.017151
393,2021090900,97,11,46163,53441,41.15,22.70,102.10,42.81,22.24,1.66,-0.46,1.722556,0.998252,3.388501,1.725573
523,2021090900,97,11,52421,53441,40.60,20.98,76.96,42.81,22.24,2.21,1.26,2.543954,0.958078,16.649010,2.655268
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3844770,2021110100,4433,36,52507,43338,22.41,26.22,217.53,21.91,26.06,-0.50,-0.16,0.524976,0.821892,34.725328,0.638741
3844151,2021110100,4433,37,37090,52585,20.75,21.61,157.61,21.04,20.88,0.29,-0.73,0.785493,0.999920,0.724041,0.785556
3844356,2021110100,4433,37,43695,42406,22.04,23.77,322.26,21.81,24.27,-0.23,0.50,0.550364,0.974222,13.037570,0.564926
3844669,2021110100,4433,37,46103,43326,22.09,20.05,166.67,22.17,17.56,0.08,-2.49,2.491285,0.979960,11.489805,2.542231


In [10]:
players = pd.read_csv("../cleaned_csv/players_cleaned.csv")
pffScoutingData = pd.read_csv("../cleaned_csv/pffScoutingData_cleaned.csv")

In [11]:
players

,nflId,height,weight,birthDate,collegeName,officialPosition,displayName
0,25511,6-4,225,1977-08-03,Michigan,QB,Tom Brady
1,28963,6-5,240,1982-03-02,"Miami, O.",QB,Ben Roethlisberger
2,29550,6-4,328,1982-01-22,Arkansas,T,Jason Peters
3,29851,6-2,225,1983-12-02,California,QB,Aaron Rodgers
4,30078,6-2,228,1982-11-24,Harvard,QB,Ryan Fitzpatrick
...,...,...,...,...,...,...,...
1674,53991,6-1,320,NaN,NaN,DT,Forrest Merrill
1675,53994,6-5,300,NaN,NaN,C,Ryan McCollum
1676,53999,6-4,312,NaN,NaN,DT,Jack Heflin
1677,54006,6-6,330,NaN,NaN,T,Jake Curhan


In [ ]:
merged_df = closest_rusher_per_blocker.merge(
    filtered_df[['gameId', 'playId', 'frameID', 'nflId','quarter','gameClock']],
    left_on=['gameId', 'playId', 'frameID', 'blocker_nflId'],
    right_on=['gameId', 'playId', 'frameID', 'nflId'],
    how='left'
).drop(columns=['nflId'])


# Add blocker_name and rusher_name from players
name_map = players[['nflId', 'displayName']].drop_duplicates()

merged_df = merged_df.merge(
    name_map.rename(columns={
        'nflId': 'blocker_nflId',
        'displayName': 'blocker_name'
    }),
    on='blocker_nflId',
    how='left'
)

merged_df = merged_df.merge(
    name_map.rename(columns={
        'nflId': 'rusher_nflId',
        'displayName': 'rusher_name'
    }),
    on='rusher_nflId',
    how='left'
)

# Add blocker_position and rusher_position from pffScoutingData using pff_positionLinedUp
pff_positions = pffScoutingData[
    ['gameId', 'playId', 'nflId', 'pff_positionLinedUp']
].drop_duplicates()

merged_df = merged_df.merge(
    pff_positions.rename(columns={
        'nflId': 'blocker_nflId',
        'pff_positionLinedUp': 'blocker_position'
    }),
    on=['gameId', 'playId', 'blocker_nflId'],
    how='left'
)

merged_df = merged_df.merge(
    pff_positions.rename(columns={
        'nflId': 'rusher_nflId',
        'pff_positionLinedUp': 'rusher_position'
    }),
    on=['gameId', 'playId', 'rusher_nflId'],
    how='left'
)

In [13]:
merged_df['rusher_position'].value_counts()

rusher_position
DRT      125124
DLT      114355
LEO       87759
LE        87426
REO       83561
ROLB      70326
LOLB      65632
RE        64810
NT        32778
NRT       18578
NLT       15293
RILB       9217
LILB       9195
MLB        3649
LLB        3202
RLB        3140
SCBL       1009
SCBR        927
SCBiL       282
SCBoL       241
SCBiR       238
LCB         219
RCB         155
SCBoR       124
SSR          12
Name: count, dtype: int64

In [ ]:
# pd.read_csv("../outputs_csv/blocker_rusher_matchups.csv")
# base_filtered.iloc[:,10:]
merged_df

,frameId,possessionTeam,defensiveTeam,yardlineSide,yardlineNumber,absoluteYardlineNumber,offenseFormation,offenseRB,offenseTE,offenseWR,...,x,y,s,a,dis,o,dir,event,frameIdEndWindow,frameIdBallSnap
0,1,TB,DAL,TB,33,43,SHOTGUN,1.0,1.0,3.0,...,37.77,24.22,0.29,0.30,0.03,165.16,84.99,NaN,36,6
1,2,TB,DAL,TB,33,43,SHOTGUN,1.0,1.0,3.0,...,37.78,24.22,0.23,0.11,0.02,164.33,92.87,NaN,36,6
2,3,TB,DAL,TB,33,43,SHOTGUN,1.0,1.0,3.0,...,37.78,24.24,0.16,0.10,0.01,160.24,68.55,NaN,36,6
3,4,TB,DAL,TB,33,43,SHOTGUN,1.0,1.0,3.0,...,37.73,24.25,0.15,0.24,0.06,152.13,296.85,NaN,36,6
4,5,TB,DAL,TB,33,43,SHOTGUN,1.0,1.0,3.0,...,37.69,24.26,0.25,0.18,0.04,148.33,287.55,NaN,36,6
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5262747,33,NYG,KC,NYG,20,30,SHOTGUN,1.0,1.0,3.0,...,40.36,44.25,7.47,0.28,0.74,48.35,60.83,NaN,37,7
5262748,34,NYG,KC,NYG,20,30,SHOTGUN,1.0,1.0,3.0,...,41.02,44.60,7.45,0.67,0.75,48.35,61.74,NaN,37,7
5262749,35,NYG,KC,NYG,20,30,SHOTGUN,1.0,1.0,3.0,...,41.68,44.94,7.48,1.43,0.75,58.99,63.79,NaN,37,7
5262750,36,NYG,KC,NYG,20,30,SHOTGUN,1.0,1.0,3.0,...,42.36,45.25,7.41,2.01,0.75,63.66,65.69,NaN,37,7


In [15]:
merged_df[(merged_df['gameId'] == 2021093000) & (merged_df['playId'] == 621)].iloc[0:50]

,gameId,playId,frameID,blocker_nflId,rusher_nflId,x_blocker,y_blocker,o_blocker,x_rusher,y_rusher,...,actual_distance,cos_theta,theta_deg,weighted_distance,quarter,gameClock,blocker_name,rusher_name,blocker_position,rusher_position
317707,2021093000,621,11,41322,43455,63.25,27.67,89.22,64.91,27.63,...,1.660482,0.999289,2.160354,1.661663,1,3:25,Brandon Linder,D.J. Reader,C,NT
317708,2021093000,621,11,41939,48145,62.99,29.64,73.93,64.68,30.84,...,2.072704,0.943761,19.307006,2.196218,1,3:25,Andrew Norwell,Wyatt Ray,LG,RE
317709,2021093000,621,11,42410,43455,62.97,26.12,64.55,64.91,27.63,...,2.458394,0.976502,12.445425,2.517552,1,3:25,A.J. Cann,D.J. Reader,RG,NT
317710,2021093000,621,11,44846,48145,62.63,31.25,79.92,64.68,30.84,...,2.090598,0.931120,21.389932,2.245251,1,3:25,Cam Robinson,Wyatt Ray,LT,RE
317711,2021093000,621,11,47818,46138,62.20,24.60,98.50,64.66,25.28,...,2.552254,0.913886,23.952011,2.792749,1,3:25,Jawaan Taylor,B.J. Hill,RT,LE
317712,2021093000,621,12,41322,43455,63.16,27.64,93.16,64.72,27.57,...,1.561570,0.999947,0.590759,1.561653,1,3:25,Brandon Linder,D.J. Reader,C,NT
317713,2021093000,621,12,41939,48145,62.97,29.73,73.93,64.52,30.86,...,1.918176,0.939554,20.023256,2.041582,1,3:25,Andrew Norwell,Wyatt Ray,LG,RE
317714,2021093000,621,12,42410,43455,62.95,26.00,69.55,64.72,27.57,...,2.365967,0.932808,21.123207,2.536393,1,3:25,A.J. Cann,D.J. Reader,RG,NT
317715,2021093000,621,12,44846,48145,62.51,31.32,77.88,64.52,30.86,...,2.061965,0.906230,25.010481,2.275321,1,3:25,Cam Robinson,Wyatt Ray,LT,RE
317716,2021093000,621,12,47818,46138,62.02,24.55,98.50,64.48,25.22,...,2.549608,0.915414,23.735409,2.785196,1,3:25,Jawaan Taylor,B.J. Hill,RT,LE


In [16]:
merged_df.columns

Index(['gameId', 'playId', 'frameID', 'blocker_nflId', 'rusher_nflId',
       'x_blocker', 'y_blocker', 'o_blocker', 'x_rusher', 'y_rusher', 'dx',
       'dy', 'actual_distance', 'cos_theta', 'theta_deg', 'weighted_distance',
       'quarter', 'gameClock', 'blocker_name', 'rusher_name',
       'blocker_position', 'rusher_position'],
      dtype='str')

In [27]:
merged_df[merged_df['gameId'] == 2021093000].drop_duplicates(subset=['gameId','playId'])

,gameId,playId,frameID,blocker_nflId,rusher_nflId,x_blocker,y_blocker,o_blocker,x_rusher,y_rusher,...,actual_distance,cos_theta,theta_deg,weighted_distance,quarter,gameClock,blocker_name,rusher_name,blocker_position,rusher_position
317085,2021093000,169,11,41322,43455,55.52,29.86,81.63,57.02,29.91,...,1.500833,0.993649,6.460848,1.510426,1,12:28,Brandon Linder,D.J. Reader,C,NT
317215,2021093000,209,11,41322,44877,57.75,23.75,95.56,58.88,23.09,...,1.308625,0.908304,24.727953,1.440735,1,11:05,Brandon Linder,Larry Ogunjobi,C,DLT
317345,2021093000,329,11,38553,43352,72.18,27.21,251.66,70.27,26.50,...,2.037695,0.999359,2.051497,2.039002,1,9:28,Riley Reiff,Adam Gotsis,RT,LE
317426,2021093000,351,12,38553,44880,72.21,27.11,266.99,70.18,26.75,...,2.061674,0.992447,7.046276,2.077364,1,9:25,Riley Reiff,Dawuane Smoot,RT,LE
317570,2021093000,599,11,41322,43455,63.41,28.24,77.58,64.85,28.31,...,1.441700,0.985888,9.636979,1.462337,1,3:35,Brandon Linder,D.J. Reader,C,NT
317707,2021093000,621,11,41322,43455,63.25,27.67,89.22,64.91,27.63,...,1.660482,0.999289,2.160354,1.661663,1,3:25,Brandon Linder,D.J. Reader,C,NT
317837,2021093000,766,11,38553,44880,84.83,29.46,286.53,83.11,31.66,...,2.792562,0.814611,35.451057,3.428091,1,0:47,Riley Reiff,Dawuane Smoot,RT,LOLB
317954,2021093000,788,11,38553,43352,85.32,29.15,254.31,82.94,27.83,...,2.721544,0.973084,13.323708,2.796824,1,0:43,Riley Reiff,Adam Gotsis,RT,DLT
318087,2021093000,893,11,41939,45226,76.28,23.80,255.02,74.42,23.13,...,1.976993,0.996449,4.829784,1.984038,2,14:21,Andrew Norwell,Josh Tupou,LG,DRT
318202,2021093000,957,11,38553,43352,35.60,20.79,76.47,37.10,21.65,...,1.729046,0.959820,16.297085,1.801428,2,14:10,Riley Reiff,Adam Gotsis,RT,DLT


In [26]:
base_filtered = pd.read_csv("../outputs_csv/base_filtered.csv")

if base_filtered.drop_duplicates(subset=['gameId','playId']).shape[0] != 7312:
    print('here')
    base_filtered = base_filtered.merge(
        merged_df[['gameId', 'playId']].drop_duplicates(),
        on=['gameId', 'playId'],
        how='inner'
    )

    base_filtered = base_filtered.merge(
        filtered_df[['gameId', 'playId', 'frameIdBallSnap']].drop_duplicates(['gameId', 'playId']),
        on=['gameId', 'playId'],
        how='left'
    )
    base_filtered = base_filtered.drop(['ttt_le_1_5_and_behind_los'],axis=1)
    base_filtered=base_filtered.rename(columns={'frameID':'frameId'})


/var/folders/rj/tzk4w23s0md7pcnpq4w2fy6c0000gn/T/ipykernel_42299/1803891794.py:1: DtypeWarning: Columns (0: foulNames, 1: foulIds) have mixed types. Specify dtype option on import or set low_memory=False.
  base_filtered = pd.read_csv("../outputs_csv/base_filtered.csv")


In [25]:
blocker_rusher_output_df=merged_df.rename(columns={'frameID':'frameId'})
blocker_rusher_output_df.to_csv("../outputs_csv/blocker_rusher_matchups.csv", index=False)
base_filtered.to_csv("../outputs_csv/base_filtered.csv", index=False)
